# MinHash と LSH で大量文書からほぼ重複を見つける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_string/minhash_lsh.ipynb)

NLP クックブック 03-23 ／ 文字列処理と正規化

数十万件の文書から、コピペや微修正による「ほぼ同じ文書」を、全ペア比較せずに高速に見つけたい。

In [1]:
!pip install -q datasketch

In [2]:
from datasketch import MinHash, MinHashLSH

def shingles(text, n=3):
    text = "".join(text.split())
    return {text[i:i + n] for i in range(len(text) - n + 1)}

def minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles(text):
        m.update(s.encode("utf-8"))
    return m

docs = {
    "a": "自然言語処理は、人間が日常的に使っている言語をコンピュータに処理させる一連の技術である。",
    "b": "自然言語処理とは、人間が日常的に使っている言語をコンピューターに処理させる技術のことである。",
    "c": "機械学習は、データから規則性を学習してタスクを解くアルゴリズムの研究分野である。",
    "d": "自然言語処理は、人間が日常的に使っている言語をコンピュータに処理させる一連の技術である！",
}
mh = {k: minhash(v) for k, v in docs.items()}

def jaccard(a, b):
    A, B = shingles(docs[a]), shingles(docs[b])
    return len(A & B) / len(A | B)

for x, y in [("a", "b"), ("a", "c"), ("a", "d")]:
    print(f"{x}-{y}: 真の Jaccard = {jaccard(x, y):.2f}  MinHash による推定 = {mh[x].jaccard(mh[y]):.2f}")

a-b: 真の Jaccard = 0.56  MinHash による推定 = 0.59
a-c: 真の Jaccard = 0.03  MinHash による推定 = 0.01
a-d: 真の Jaccard = 0.95  MinHash による推定 = 0.97


In [3]:
lsh = MinHashLSH(threshold=0.5, num_perm=128)
for k, m in mh.items():
    lsh.insert(k, m)
for k in docs:
    print(k, "と似ている:", sorted(set(lsh.query(mh[k])) - {k}))

a と似ている: ['b', 'd']
b と似ている: ['a', 'd']
c と似ている: []
d と似ている: ['a', 'b']


- MinHash は集合の Jaccard 係数を固定長の署名で近似する。LSH は署名が似たものだけを同じバケットに入れ、候補を絞る。
- 大規模な事前学習コーパスの重複除去で標準的に使われている。意味的な重複 (言い換え) は拾えないので、それには埋め込みを使う。